# Independent Q-Learning (IQL) — Demo Notebook

**Project:** Predator-Prey Archetype Gridworld  
**Algorithm:** Tabular Independent Q-Learning  
**Audience:** Students new to reinforcement learning and this codebase

---

## What You Will Learn

By the end of this notebook you will be able to:

1. Create and step through the gridworld environment manually
2. Understand what an observation and a reward look like at the code level
3. Explain the Q-learning update rule and trace it through the IQL source code
4. Train an IQL agent and interpret its learning curves
5. Evaluate a trained agent and compare it against a random baseline

## Prerequisites

- Python basics (functions, dicts, loops)
- A rough idea of what reinforcement learning is — agent, environment, reward  
- No prior MARL knowledge required

## How to Run

Run cells **top to bottom** in order. Each cell builds on the previous one.  
The project must be installed first: `pip install -e ..` from the repo root.

---
## Part 1 — Setup

We add `src/` to the Python path so the project modules are importable, then bring in everything we need.

In [ ]:
import sys
from pathlib import Path

# Make src/ importable regardless of where you launched the kernel from
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)
print("Python path (first entry):", sys.path[0])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import logging
from collections import defaultdict

from multi_agent_package.core.gridworld import GridWorldEnv
from multi_agent_package.core.agent import Agent
from baselines.IQL.iql import IQL

# Set up logging so training messages appear in the notebook output
logging.basicConfig(
    level=logging.INFO,
    format="[ep %(message)s",
)

print("All imports OK")

---
## Part 2 — The Environment

### 2.1 What Is the Gridworld?

The environment is a discrete 2D grid. At each timestep:
- Every agent picks one of 5 actions: `Right(0)`, `Up(1)`, `Left(2)`, `Down(3)`, `Noop(4)`
- All agents move **simultaneously**
- If a predator lands on the same cell as a prey → **capture** → episode may end

The architecture has four independently swappable layers:

```
Environment dynamics   →   Perception (observations)   →   Incentives (rewards)   →   Learning
      (core)                   (observations/)                 (rewards/)             (baselines/)
```

We build the environment by hand first to understand each piece.

### 2.2 Creating Agents and the Environment

In [ ]:
# Create a simple 1v1 environment: 1 predator vs 1 prey on a 5x5 grid
# This is the smallest setup where you can see the core dynamics.

agents = [
    Agent(agent_name="predator_1", agent_team="predator_1", agent_type="predator"),
    Agent(agent_name="prey_1",     agent_team="prey_1",     agent_type="prey"),
]

env = GridWorldEnv(
    agents=agents,
    size=5,                    # 5x5 grid
    perc_num_obstacle=10.0,    # ~10% of cells are obstacles
    render_mode=None,          # headless — no pygame window
    seed=42,                   # fixed seed for reproducibility
    capture_threshold=1,       # episode ends on first capture
    max_steps=100,             # or after 100 steps
)

print("Environment created")
print(f"  Grid size       : {env.size} x {env.size}")
print(f"  Agents          : {[ag.agent_name for ag in env.agents]}")
print(f"  Capture threshold: {env.capture_threshold}")
print(f"  Max steps       : {env.max_steps}")

### 2.3 Resetting the Environment

`env.reset()` places agents and obstacles at random positions (reproducible with a fixed seed) and returns the first observation.

In [ ]:
obs, info = env.reset(seed=42)

print("Observation keys (one per agent):")
for agent_name, agent_obs in obs.items():
    print(f"  {agent_name}: {agent_obs}")

print("\nAgent positions after reset:")
for ag in env.agents:
    print(f"  {ag.agent_name}: {ag._agent_location}")

print(f"\nNumber of obstacles: {len(env._obstacle_location)}")

### 2.4 Visualising the Grid

Since we're running headless (no pygame), we use matplotlib to draw the current state.

In [ ]:
def render_grid(env, ax=None, title="Gridworld State"):
    """Draw the current grid state: obstacles (black), predators (red), prey (blue)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    
    size = env.size
    ax.set_xlim(0, size)
    ax.set_ylim(0, size)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=12)
    
    # Background
    ax.set_facecolor("#f8f9fa")
    
    # Grid lines
    for i in range(size + 1):
        ax.axhline(i, color="#cccccc", linewidth=0.8)
        ax.axvline(i, color="#cccccc", linewidth=0.8)
    
    # Obstacles
    for obs_pos in env._obstacle_location:
        x, y = obs_pos
        ax.add_patch(mpatches.Rectangle((x, y), 1, 1, color="#2c3e50", zorder=2))
    
    # Agents
    for ag in env.agents:
        x, y = ag._agent_location
        cx, cy = x + 0.5, y + 0.5
        if ag.agent_type.startswith("predator"):
            color, marker, label = "#e74c3c", "v", "P"
        else:
            color, marker, label = "#2980b9", "^", "R"
        ax.plot(cx, cy, marker=marker, markersize=22, color=color, zorder=3)
        ax.text(cx, cy, label, ha="center", va="center",
                fontsize=9, color="white", fontweight="bold", zorder=4)
    
    ax.set_xticks(range(size))
    ax.set_yticks(range(size))
    ax.set_xticklabels(range(size), fontsize=8)
    ax.set_yticklabels(range(size), fontsize=8)
    
    # Legend
    legend = [
        mpatches.Patch(color="#e74c3c", label="Predator (P)"),
        mpatches.Patch(color="#2980b9", label="Prey (R)"),
        mpatches.Patch(color="#2c3e50", label="Obstacle"),
    ]
    ax.legend(handles=legend, loc="upper right", fontsize=8)
    return ax

render_grid(env, title="Initial State (seed=42)")
plt.tight_layout()
plt.show()

### 2.5 Stepping the Environment

`env.step(actions)` takes a dict mapping each agent name to an integer action.

**Action map:** `0=Right, 1=Up, 2=Left, 3=Down, 4=Noop`  
*(Note: in this coordinate system, Y increases upward — action 1 moves to a higher Y value)*

The return value is a dict — **not** the standard Gymnasium tuple:

In [ ]:
obs, _ = env.reset(seed=42)

print("Before step:")
for ag in env.agents:
    print(f"  {ag.agent_name} at {ag._agent_location}")

# Both agents move right
actions = {"predator_1": 0, "prey_1": 2}  # predator moves right, prey moves left
result = env.step(actions)

print("\nAfter step:")
for ag in env.agents:
    print(f"  {ag.agent_name} at {ag._agent_location}")

print("\nStep result keys:")
for key, val in result.items():
    print(f"  {key}: {val}")

### 2.6 Understanding Observations

This environment uses the `LocalRadiusObservation` builder by default. It gives each agent:
- `local` — the agent's own grid position
- `visible_agents` — nearby agents within Manhattan radius r  
- `visible_obstacles` — nearby obstacles within radius r

Let's attach the observation builder and inspect the output:

In [ ]:
from multi_agent_package.registry import get_observation_builder

# Attach a LocalRadius observation builder with radius=3
obs_builder = get_observation_builder("local_radius", radius=3,
                                       include_agents=True, include_obstacles=True)
env.observation_builder = obs_builder.build

obs, _ = env.reset(seed=42)

print("Observation for predator_1:")
pred_obs = obs["predator_1"]
print(f"  Own position     : {pred_obs['local']}")
print(f"  Visible agents   : {pred_obs['visible_agents']}")
print(f"  Visible obstacles: {list(pred_obs['visible_obstacles'].keys())[:3]} ...")
print(f"  Radius           : {pred_obs['radius']}")

### 2.7 Understanding Rewards

The reward system is also pluggable. The base reward (hardcoded in `gridworld.py`) is:

| Event | Reward |
|-------|--------|
| Predator captures prey | +100 (predator), −100 (prey) |
| Each step | −5 (predator only — step cost) |
| Agent on obstacle | −200 |

Let's attach a reward function and see the values:

In [ ]:
from multi_agent_package.registry import get_reward_function

# Combine: base reward + predator distance shaping
base_rf = get_reward_function("base")
dist_rf = get_reward_function("predator_distance", weight=0.5)

def combined_reward(env_inst):
    total = {ag.agent_name: 0.0 for ag in env_inst.agents}
    for rf in [base_rf, dist_rf]:
        for k, v in rf.compute(env_inst).items():
            total[k] += v
    return total

env.reward_fn = combined_reward

# Take a step and inspect rewards
obs, _ = env.reset(seed=42)
result = env.step({"predator_1": 0, "prey_1": 2})

print("Rewards after one step:")
for agent_name, reward in result["reward"].items():
    print(f"  {agent_name}: {reward:.2f}")

print("\nExplanation:")
print("  predator_1: −5 (step cost) + distance shaping (negative distance to prey)")
print("  prey_1    : 0 (no base reward unless captured, no shaping for prey in this config)")

---
## Part 3 — Independent Q-Learning (IQL)

### 3.1 The Core Idea

Q-learning is a **value-based** reinforcement learning algorithm. It learns a function:

$$Q(s, a) \approx \text{expected total discounted reward from state } s, \text{ taking action } a$$

Once $Q$ is learned, the optimal policy is simply: always take the action with the highest Q-value:

$$\pi^*(s) = \arg\max_a Q(s, a)$$

**The Bellman update** — how Q-values are updated after each step:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \Big]$$

Where:
- $\alpha$ = learning rate (how fast to update)
- $\gamma$ = discount factor (how much future rewards matter)
- $r$ = reward received
- $s'$ = next state
- The bracketed term is the **TD error** (temporal difference error)

### 3.2 What Makes It "Independent"?

In a multi-agent environment, each agent could theoretically condition its Q-function on every other agent's observation. IQL **doesn't do this** — each agent maintains its **own independent Q-table** and treats other agents as part of the environment.

This is simple and scalable, but it violates the stationarity assumption: from agent A's perspective, the "environment" changes as agent B learns, because B's policy is evolving. IQL has no formal convergence guarantee in multi-agent settings, but it often works well in practice on small environments.

### 3.3 IQL in the Codebase — Walkthrough

Let's create an IQL instance and trace through each key method.

In [ ]:
# Fresh environment for clean training
agents = [
    Agent(agent_name="predator_1", agent_team="predator_1", agent_type="predator"),
    Agent(agent_name="prey_1",     agent_team="prey_1",     agent_type="prey"),
]
env = GridWorldEnv(agents=agents, size=5, perc_num_obstacle=10.0,
                   render_mode=None, seed=42, capture_threshold=1, max_steps=100)
env.observation_builder = obs_builder.build
env.reward_fn = combined_reward

config = {
    "alpha":         0.1,    # learning rate
    "gamma":         0.99,   # discount factor
    "epsilon":       1.0,    # start with full exploration
    "epsilon_decay": 0.995,  # decay epsilon each episode
    "min_epsilon":   0.05,   # never go below 5% exploration
    "action_dim":    5,      # 5 possible actions
    "episodes":      300,    # training episodes
    "seed":          42,
}

algo = IQL(env, config)

print("IQL created")
print(f"  Agents discovered: {algo.agent_ids}")
print(f"  Q-tables          : one per agent, starts empty (defaultdict)")
print(f"  Action dim        : {algo.action_dim}")
print(f"  Epsilon           : {algo.epsilon}")

In [ ]:
# --- State encoding ---
# Q-tables are indexed by (state, action). The state must be hashable.
# IQL recursively converts the observation dict into a nested tuple.

obs, _ = env.reset(seed=42)
state = algo._encode_state(obs["predator_1"])

print("Raw observation (dict):")
print(f"  {obs['predator_1']}")
print()
print("Encoded state (hashable tuple) — first 100 chars:")
print(f"  {str(state)[:100]} ...")
print()
print("This tuple is used as the Q-table key.")
print(f"Q-table entry for this state: {algo.q_tables['predator_1'][state]}")
print("(All zeros at the start — no experience yet)")

In [ ]:
# --- Action selection ---
# With epsilon=1.0 (full exploration), every action is random.
# As epsilon decays, actions are taken greedily: argmax Q(s, a)

obs, _ = env.reset(seed=42)
actions = algo.select_actions(obs)
print(f"Epsilon = {algo.epsilon:.2f} → actions are random: {actions}")

# Temporarily set epsilon to 0 to show greedy selection
algo.epsilon = 0.0
actions = algo.select_actions(obs)
print(f"Epsilon = {algo.epsilon:.2f} → actions are greedy : {actions}")
print("  (Still 0s because Q-table is all zeros — ties broken by argmax picking index 0)")

# Reset epsilon for training
algo.epsilon = 1.0

In [ ]:
# --- Manual Q-update ---
# Let's trace one Bellman update step by step.

obs, _ = env.reset(seed=42)
actions = algo.select_actions(obs)
result = env.step(actions)

agent_id = "predator_1"
s      = algo._encode_state(obs[agent_id])
a      = actions[agent_id]
r      = float(result["reward"][agent_id])
s_next = algo._encode_state(result["obs"][agent_id])
done   = result["terminated"] or result["truncated"]

q_current  = algo.q_tables[agent_id][s][a]
q_next_max = 0.0 if done else float(np.max(algo.q_tables[agent_id][s_next]))
td_error   = r + algo.gamma * q_next_max - q_current
new_q      = q_current + algo.alpha * td_error

print("Bellman update for predator_1:")
print(f"  Action taken      : {a}  (0=Right, 1=Up, 2=Left, 3=Down, 4=Noop)")
print(f"  Reward received   : {r:.2f}")
print(f"  Q(s, a) before    : {q_current:.4f}")
print(f"  max Q(s', a')     : {q_next_max:.4f}")
print(f"  TD error          : {r} + {algo.gamma}×{q_next_max:.4f} − {q_current:.4f} = {td_error:.4f}")
print(f"  Q(s, a) after     : {q_current:.4f} + {algo.alpha}×{td_error:.4f} = {new_q:.4f}")

---
## Part 4 — Training

We train by running episodes and applying the Bellman update at each step.  
We collect metrics manually so we can plot them afterward.

In [ ]:
def train_and_collect(env, config, n_episodes=300, log_every=50):
    """Train IQL and return the algorithm and episode metrics."""
    algo = IQL(env, config)
    
    rewards_per_episode = {aid: [] for aid in algo.agent_ids}
    episode_lengths     = []
    epsilon_history     = []
    capture_flags       = []  # 1 if episode ended with a capture, else 0
    
    for ep in range(n_episodes):
        obs, _ = env.reset()
        done = False
        ep_reward = {aid: 0.0 for aid in algo.agent_ids}
        steps = 0
        
        while not done:
            actions  = algo.select_actions(obs)
            result   = env.step(actions)
            next_obs = result["obs"]
            rewards  = result["reward"]
            done     = result["terminated"] or result["truncated"]
            
            # Bellman update for every agent
            for aid in algo.agent_ids:
                s      = algo._encode_state(obs[aid])
                a      = actions[aid]
                r      = float(rewards[aid])
                s_next = algo._encode_state(next_obs[aid])
                
                q_curr     = algo.q_tables[aid][s][a]
                q_next_max = 0.0 if done else float(np.max(algo.q_tables[aid][s_next]))
                td         = r + algo.gamma * q_next_max - q_curr
                algo.q_tables[aid][s][a] += algo.alpha * td
                
                ep_reward[aid] += r
            
            obs    = next_obs
            steps += 1
        
        # Decay epsilon
        algo.epsilon = max(algo.min_epsilon, algo.epsilon * algo.epsilon_decay)
        
        # Record metrics
        for aid in algo.agent_ids:
            rewards_per_episode[aid].append(ep_reward[aid])
        episode_lengths.append(steps)
        epsilon_history.append(algo.epsilon)
        capture_flags.append(1 if result["terminated"] else 0)
        
        if (ep + 1) % log_every == 0:
            avg_len = np.mean(episode_lengths[-log_every:])
            cap_rate = np.mean(capture_flags[-log_every:]) * 100
            print(f"  Episode {ep+1:4d}/{n_episodes} | "
                  f"ε={algo.epsilon:.3f} | "
                  f"avg steps={avg_len:.1f} | "
                  f"capture rate={cap_rate:.1f}%")
    
    metrics = {
        "rewards":      rewards_per_episode,
        "lengths":      episode_lengths,
        "epsilon":      epsilon_history,
        "capture_rate": capture_flags,
    }
    return algo, metrics


print("Training IQL for 300 episodes on a 5×5 grid (1 predator vs 1 prey)...")
trained_iql, metrics = train_and_collect(env, config, n_episodes=300, log_every=50)

### 4.1 Training Curves

Three signals tell us whether learning is happening:
- **Predator reward** — should increase over time (capturing prey more often)
- **Episode length** — should decrease (captures happen faster)
- **Epsilon** — decreases by design; confirms the schedule is working

In [ ]:
def smooth(x, window=20):
    """Rolling mean for cleaner curves."""
    return np.convolve(x, np.ones(window)/window, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("IQL Training — 5×5 grid, 1 predator vs 1 prey", fontsize=13)

# Predator reward
ax = axes[0]
r = metrics["rewards"]["predator_1"]
ax.plot(r, alpha=0.3, color="#e74c3c", label="raw")
ax.plot(np.arange(len(smooth(r))), smooth(r), color="#c0392b", linewidth=2, label="smoothed (w=20)")
ax.set_xlabel("Episode")
ax.set_ylabel("Total reward")
ax.set_title("Predator Reward")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Episode length
ax = axes[1]
l = metrics["lengths"]
ax.plot(l, alpha=0.3, color="#27ae60", label="raw")
ax.plot(np.arange(len(smooth(l))), smooth(l), color="#1e8449", linewidth=2, label="smoothed")
ax.set_xlabel("Episode")
ax.set_ylabel("Steps")
ax.set_title("Episode Length (lower = faster capture)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Epsilon
ax = axes[2]
ax.plot(metrics["epsilon"], color="#8e44ad", linewidth=2)
ax.set_xlabel("Episode")
ax.set_ylabel("Epsilon")
ax.set_title("Exploration Rate (ε-greedy)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
first_50  = np.mean(metrics["lengths"][:50])
last_50   = np.mean(metrics["lengths"][-50:])
cap_rate  = np.mean(metrics["capture_rate"]) * 100
print(f"\nSummary:")
print(f"  Avg episode length (first 50 eps): {first_50:.1f} steps")
print(f"  Avg episode length (last  50 eps): {last_50:.1f} steps")
print(f"  Overall capture rate             : {cap_rate:.1f}%")

### 4.2 Inspecting the Q-Table

After training, the Q-table maps states to action values. Let's look at how many unique states were visited and what the Q-values look like.

In [ ]:
# How big is the Q-table?
for aid in trained_iql.agent_ids:
    n_states = len(trained_iql.q_tables[aid])
    print(f"{aid}: {n_states} unique states visited")

# Show the Q-values for one visited state
pred_table = trained_iql.q_tables["predator_1"]
# Find the state with the most extreme max Q-value (most "confident" action)
best_state = max(pred_table.keys(), key=lambda s: np.max(pred_table[s]))
q_vals = pred_table[best_state]

fig, ax = plt.subplots(figsize=(6, 3))
action_labels = ["Right(0)", "Up(1)", "Left(2)", "Down(3)", "Noop(4)"]
colors = ["#e74c3c" if v == max(q_vals) else "#bdc3c7" for v in q_vals]
ax.bar(action_labels, q_vals, color=colors)
ax.set_title("Q-values for best-known state (predator_1)\nRed bar = greedy action", fontsize=10)
ax.set_ylabel("Q-value")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print(f"\nGreedy action in this state: {np.argmax(q_vals)} ({action_labels[np.argmax(q_vals)]})")

---
## Part 5 — Evaluation

We evaluate by running episodes with **epsilon=0** (fully greedy — no exploration) and comparing against a **random agent baseline**.

In [ ]:
def evaluate_agent(env, algo, n_episodes=100, epsilon_override=0.0):
    """Run n_episodes with frozen policy and return capture rate and avg length."""
    original_eps = algo.epsilon
    algo.epsilon = epsilon_override
    
    capture_count = 0
    lengths       = []
    
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done   = False
        steps  = 0
        
        while not done:
            actions = algo.select_actions(obs)
            result  = env.step(actions)
            obs     = result["obs"]
            done    = result["terminated"] or result["truncated"]
            steps  += 1
        
        if result["terminated"]:
            capture_count += 1
        lengths.append(steps)
    
    algo.epsilon = original_eps
    return {
        "capture_rate": capture_count / n_episodes * 100,
        "avg_length":   np.mean(lengths),
    }


# Trained agent (greedy)
trained_stats = evaluate_agent(env, trained_iql, n_episodes=100, epsilon_override=0.0)

# Random agent (epsilon=1.0)
random_stats  = evaluate_agent(env, trained_iql, n_episodes=100, epsilon_override=1.0)

print("Evaluation over 100 episodes:")
print(f"  {'':25s} {'Capture Rate':>15s} {'Avg Episode Length':>20s}")
print(f"  {'-'*60}")
print(f"  {'Trained IQL (greedy)':25s} {trained_stats['capture_rate']:>14.1f}% {trained_stats['avg_length']:>20.1f}")
print(f"  {'Random (ε=1.0)':25s} {random_stats['capture_rate']:>14.1f}% {random_stats['avg_length']:>20.1f}")

---
## Part 6 — Save and Load

Trained Q-tables can be persisted and reloaded. This is essential for reproducibility — you train once, save, and can evaluate later without retraining.

In [ ]:
import os

save_path = str(PROJECT_ROOT / "notebooks" / "iql_demo_checkpoint.pkl")
trained_iql.save(save_path)
print(f"Saved to: {save_path}")
print(f"File size: {os.path.getsize(save_path) / 1024:.1f} KB")

# Reload
loaded_iql = IQL.load(env, config, save_path)

# Verify the loaded model gives the same actions on the same state
obs, _ = env.reset(seed=99)
loaded_iql.epsilon = 0.0
trained_iql.epsilon = 0.0

actions_trained = trained_iql.select_actions(obs)
actions_loaded  = loaded_iql.select_actions(obs)
print(f"\nActions from trained : {actions_trained}")
print(f"Actions from loaded  : {actions_loaded}")
print(f"Match: {actions_trained == actions_loaded}")

---
## Part 7 — Experiment: Effect of Epsilon Decay

Epsilon decay controls the **exploration-exploitation trade-off**. Too fast → agent exploits too early before learning enough. Too slow → agent wastes time exploring a policy it already knows is good.

In [ ]:
decay_values = [0.99, 0.995, 0.999]
results = {}

for decay in decay_values:
    print(f"Training with epsilon_decay={decay}...")
    cfg = {**config, "epsilon_decay": decay, "epsilon": 1.0}
    _, m = train_and_collect(env, cfg, n_episodes=300, log_every=300)  # silent
    results[decay] = m

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Effect of Epsilon Decay on IQL Performance", fontsize=13)

colors = ["#e74c3c", "#2ecc71", "#3498db"]
for (decay, m), color in zip(results.items(), colors):
    label = f"decay={decay}"
    
    # Episode length
    l = m["lengths"]
    axes[0].plot(smooth(l, 20), color=color, linewidth=1.8, label=label)
    
    # Epsilon
    axes[1].plot(m["epsilon"], color=color, linewidth=1.8, label=label)

axes[0].set_title("Episode Length (lower = faster captures)")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Steps (smoothed)")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Epsilon Schedule")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Epsilon")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Summary

In this notebook you:

| Step | What happened |
|------|---------------|
| Created the environment | `GridWorldEnv` + `Agent` objects, headless mode |
| Stepped manually | Saw how `env.step()` returns a dict with obs/reward/terminated/truncated |
| Understood observations | `LocalRadiusObservation` gives each agent its position and nearby entities |
| Understood rewards | Base reward + shaping, both pluggable |
| Traced the Q-update | Bellman equation applied step by step |
| Trained IQL for 300 episodes | Watched reward, length, and epsilon evolve |
| Evaluated | Trained agent outperforms random baseline |
| Saved and loaded | Checkpoints are exact |
| Ran an ablation | Epsilon decay speed changes convergence rate |

## Exercises

1. **Scale up:** Change `size=8` and `agents` to 2 predators vs 2 prey. How does the episode length change?
2. **Reward ablation:** Remove `dist_rf` (distance shaping) from `combined_reward`. Does the predator still learn to capture?
3. **Learning rate:** Try `alpha=0.01` and `alpha=0.5`. Which converges faster? Which is more stable?
4. **Observation ablation:** Swap `local_radius` for `local_only` (blind agent). Does the predator still capture prey?
5. **State space:** Print `len(trained_iql.q_tables['predator_1'])` at episodes 50, 100, 200, 300. How fast does the table grow? What happens as the grid gets bigger?

---
*Next notebook: `02_cql_demo.ipynb` — Centralized Q-Learning, joint state-action spaces, and how coordination emerges.*